# Phase 2A - Diagnostic framework (Exercise 1)

**Phase 2A - Diagnostic framework & Exercise 1 (diagnose).** Independent notebook - runs standalone in Colab or locally (offline, no key).

## The 3-probe framework
- **Probe 1 retrieval:** precision/recall/F1 of retrieved evidence vs gold.
- **Probe 2 utilization:** 3-arm contrastive reader (no-context / oracle / provider) - `util != hit`.
- **Probe 3 failure root-cause:** retrieval_miss / partial / retrieved-but-unused.

## 0. Setup (self-contained)

In [ ]:
# Self-contained setup - works standalone in Google Colab or locally.
import sys, os, subprocess
from pathlib import Path
REPO_URL = "https://github.com/syaikhipin/kdd26-memdiag"
try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
if IN_COLAB:
    repo = Path("/content/kdd26-memdiag")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=False)
    SOURCE = repo / "experiment" / "github_submission" / "source"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "matplotlib", "pyyaml"], check=False)
else:
    SOURCE = None
    for cand in [Path.cwd(), *Path.cwd().parents]:
        for sub in ("source", "experiment"):
            if (cand / sub / "run.py").exists():
                SOURCE = cand / sub
                break
        if SOURCE:
            break
    if SOURCE is None:
        raise FileNotFoundError("Run from the repo root (or in Colab it auto-clones).")
sys.path.insert(0, str(SOURCE))
os.environ.setdefault("OPENAI_BASE_URL", "https://api.openai.com/v1")
SOURCE_DIR = SOURCE
PROJECT_ROOT = SOURCE.parent
RESULTS_DIR = PROJECT_ROOT / "results"
print("SOURCE_DIR =", SOURCE, "| IN_COLAB =", IN_COLAB)

print("SOURCE_DIR =", SOURCE_DIR)

## 📖 Narrative: Why diagnose before benchmarking?

Before we compare providers, we need to know **where** memory fails. The 3-probe framework
separates three failure modes that are often conflated:

- **Retrieval failure** — the right evidence was never retrieved (precision/recall)
- **Utilization failure** — the evidence was retrieved but the agent didn't use it
- **Storage failure** — the memory representation lost critical information

This distinction matters because the fix is different for each: retrieval needs better search,
utilization needs better prompting/grounding, storage needs better representation.

## Exercise 1 - memory failure diagnosis

In [ ]:
import subprocess
cmd = [sys.executable,'-m','diagnostic_framework','diagnose',
 '--strategies','verbatim,extracted_facts,episodic,hybrid',
 '--probes','relevance,utilization,failure','--top_k','5','--max-questions','20']
r = subprocess.run(cmd, cwd=str(SOURCE_DIR), capture_output=True, text=True)
print(r.stdout[-2200:])
if r.returncode: print('STDERR:', r.stderr[-800:])

## 📖 Narrative: Reading the diagnostic table

Look at the `util` column vs `hit` column. If `util < hit`, the system retrieved the right
evidence but the agent couldn't use it — a **utilization failure**. If `util > hit`, the
context mentioned the answer even though the exact gold turn wasn't retrieved — meaning
**other turns carried the information**. The gap between `util` and `hit` is the
**utilization diagnostic signal**.

### 📝 Quick Quiz
1. **Which strategy has the highest retrieval hit on LoCoMo?** (Check the `hit` column)
2. **Is `util` ever different from `hit`?** Which strategy shows the biggest gap?
3. **What does `util=0.300, hit=0.300` mean?** Did the agent use the memory or not?

**Read it:** is `util` ever different from `hit`? That gap is the retrieval-vs-utilization failure.